# Time Pattern Generator (20 kHz Biphasic Waveforms)

This notebook generates **stimulation time-patterns** for each `(CV, Frequency)` pair using:

- **Slack reference patterns** (loaded from `.txt` files), and  
- **CV=0:** one exact-periodic representation copied across all 100 historical repetition labels
- **CV>0:** one Slack reference plus 99 gamma-distributed generated patterns

All outputs share:
- **Sampling rate:** 20 kHz (50 µs)  
- **Waveform length:** 3 s total  
  - 2 s stimulation  
  - 1 s post-stim silence  
- **Pulse shape:** biphasic ±1 µA (4 samples → 1 sample → 4 samples)  
- **Waveform type:** `float32`, shape `(60001,)`

---

## Slack Patterns

Slack waveforms are provided as `.txt` files containing:

```
time (s), amplitude (A)
```

Each Slack waveform is saved as:

```
rep001_slack
```

Required filename pattern:

```
Pattern_{CV:.1f}cv_2s_{FREQ}Hz_100uA.txt
```

Example:
```
Pattern_0.5cv_2s_100Hz_100uA.txt
```

---

## Generated Patterns

Each `(cv, freq)` pair includes **99 generated waveforms**:

- **CV = 0.0** → one exact-periodic spike train for every repetition label  
  (`rep001_slack` is retained only as a compatibility label)

- **CV > 0.0** → gamma-distributed IPIs  
  Keys: `repXXX_gamma`

All generated spikes are rendered as biphasic pulses on the 20 kHz grid.

Example keys:
```
rep002_gamma  … rep100_gamma      (CV > 0)
rep002_regular … rep100_regular   (CV = 0)
```

---

## Saved Output Files

Two files are generated per `(cv, freq)` pair:

### 1. Waveform bundle (`.npz`)
```
cv{CV:.1f}_f{FREQ}Hz_waveforms.npz
```

Contains:
- CV=0: 100 copies of one generated exact-periodic waveform
- CV>0: 1 Slack waveform (`rep001_slack`) plus 99 generated waveforms

Each stored as a float32 array of length **60001**.

### 2. Metadata (`.json`)
```
cv{CV:.1f}_f{FREQ}Hz_meta.json
```

Includes:
- CV, frequency  
- sampling rate  
- stim/silence durations  
- pulse parameters  
- number of waveforms  
- path to `.npz` file  

---

## Manifest

A global CSV file:
```
manifest.csv
```

Contains one row per `(cv, freq)` pair:
```
cv, freq_hz, npz, meta
```

Used for downstream automation and analysis.

---


In [ ]:
# ---------------- User Inputs ----------------

# CV values and frequencies you want to generate patterns for
cv_vals   = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
freq_vals = [10, 20, 50, 100, 200]   # Hz

# Pulse width & stimulus duration (seconds)
# The biphasic template below is fixed at 4-1-4 samples at 20 kHz (~450 µs total).
pw = 450e-6            # seconds (currently informational, not changing template)
stim_duration = 2.0    # seconds of stimulation

# Path to Slack .txt files
# Example expected files:
#   ./data/time-patterns/slack/Pattern_0.1cv_2s_10Hz_100uA.txt
#   ./data/time-patterns/slack/Pattern_1.0cv_2s_200Hz_100uA.txt
from pathlib import Path
_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / 'data/time-patterns/slack').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from inside scs-periodicity')
slack_base = str(REPO_ROOT / 'data/time-patterns/slack')

# Output folder for generated bundles (per (CV, freq))
out_dir = str(REPO_ROOT / 'data/time-patterns/population')

# Parallel execution flag:
#   True  -> use all available CPU cores via ProcessPoolExecutor
#   False -> run serially (useful for debugging)
parallel = True

# Global random seed for reproducibility across the whole run
base_seed = 123


In [ ]:
# --------------- Import Packages --------------- #

from __future__ import annotations
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import gc
import json
import csv
import glob
from dataclasses import dataclass
from typing import Dict, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# --------------- Helper Functions --------------- #

@dataclass
class StimSpec:
    """
    Specification of the stimulation waveform and sampling grid.

    Attributes
    ----------
    fs : float
        Sampling frequency in Hz (20,000 Hz => 50 µs per sample).
    stim_dur_s : float
        Duration of the stimulation window in seconds (e.g., 2.0 s).
    post_silence_s : float
        Silent period appended after stimulation (e.g., 1.0 s).
    amp_uA : float
        Absolute amplitude for each phase of the biphasic pulse (±amp_uA).
    neg_samples : int
        Number of samples in the negative phase (first phase).
    interphase_samples : int
        Number of zero-amplitude samples between negative and positive phases.
    pos_samples : int
        Number of samples in the positive phase (second phase).
    """
    fs: float = 20_000.0
    stim_dur_s: float = 2.0
    post_silence_s: float = 1.0
    amp_uA: float = 1.0
    neg_samples: int = 4
    interphase_samples: int = 1
    pos_samples: int = 4

    @property
    def dt(self) -> float:
        """Sampling interval in seconds (e.g., 50 µs)."""
        return 1.0 / self.fs
    
    @property
    def pulse_samples(self) -> int:
        """Number of samples in a single stim pulse (neg + inter + pos)"""
        return self.neg_samples + self.interphase_samples + self.pos_samples
    
    @property
    def min_ipi_s(self) -> float:
        """Minimum interpulse interval (seconds) to safeguard against overlap"""
        return self.pulse_samples / self.fs

    @property
    def stim_samples(self) -> int:
        """Number of samples in the stimulation window [0:2] seconds: ((2 s * 20 kHz) + 1 = 40,001)."""
        return int(round(self.stim_dur_s * self.fs) + 1)

    @property
    def total_samples(self) -> int:
        """Total samples (stim + silence). For 2 s stim + 1 s silence => 60,001 samples."""
        return int(round((self.stim_dur_s + self.post_silence_s) * self.fs) + 1)
    
def slack_path_format(cv: float, freq: float | int, base_dir: str) -> str:
    """
    Build the expected Slack filename for a given (cv, freq) pair.

    File pattern: Pattern_{cv:.1f}cv_2s_{freq}Hz_100uA.txt

    Examples
    --------
    cv=0.5, freq=200 -> Pattern_0.5cv_2s_200Hz_100uA.txt
    cv=1.0, freq=50  -> Pattern_1.0cv_2s_50Hz_100uA.txt
    """
    cv_str = f"{cv:.1f}"
    fname = f"Pattern_{cv_str}cv_2s_{int(freq)}Hz_100uA.txt"
    return os.path.join(base_dir, fname)

def load_slack_waveform_1uA(txt_path: str, spec: StimSpec) -> np.ndarray:
    """
    Load a Slack `.txt` file that already encodes the full waveform as (time (s), amplitude (A)),
    and converts the 100 uA pulses to 1 uA pulses.

    Parameters
    ----------
    txt_path : str
        Path to the file. The file must have at least 2 columns:
          - column 0: time in seconds
          - column 1: amplitude in microamps
    spec : StimSpec
        Stimulation/sampling specification (20 kHz, 2 s stim, 1 s silence).

    Returns
    -------
    y : np.ndarray
        1D array of length spec.total_samples containing the waveform:
        - Stimulus region (first spec.stim_samples) is filled using
          the Slack amplitudes (±0.0001 A) converted to ±1 uA.
        - Post-silence region is left at 0.
    """
    # Load raw data from text file
    arr = np.loadtxt(txt_path, delimiter=',', dtype=float)

    # Expect shape (N, 2) at minimum: [time, amplitude]
    if arr.ndim != 2 or arr.shape[1] < 2:
        raise ValueError(f"Slack txt must have 2 columns (time, amplitude). Got shape {arr.shape}.")

    # Extract amplitude arrays
    a = arr[:, 1].astype(np.float32) # amplitude in A (±0.0001)

    # Convert to ±1 uA
    return a*10000.0

def pulse_start_indices_from_times(spike_times_s: np.ndarray, spec: StimSpec) -> np.ndarray:
    """Map continuous pulse times to the exact sample starts used by waveform generation."""
    times = np.asarray(spike_times_s, dtype=float)
    idx = np.floor(times * float(spec.fs)).astype(int)
    return idx[(idx >= 0) & (idx + spec.pulse_samples <= spec.stim_samples)]


def validate_pulse_start_indices(idx: np.ndarray, expected_n: int, spec: StimSpec) -> None:
    """Fail if generated pulses are not exact-count, unique, in-window, and non-overlapping."""
    idx = np.asarray(idx, dtype=int)
    if idx.size != int(expected_n):
        raise ValueError(f"Expected {expected_n} pulses, got {idx.size} after sample-grid placement.")
    if np.unique(idx).size != idx.size:
        raise ValueError("Pulse starts collapsed onto duplicate sample indices.")
    if idx.size and (idx[0] < 0 or idx[-1] + spec.pulse_samples > spec.stim_samples):
        raise ValueError("At least one pulse does not fit fully inside the stimulation window.")
    if idx.size > 1 and np.any(np.diff(idx) < spec.pulse_samples):
        raise ValueError(f"Pulse starts violate the {spec.pulse_samples}-sample minimum IPI.")


def count_negative_phase_onsets(waveform: np.ndarray, eps: float = 0.0) -> int:
    """Count pulse onsets from 0 -> negative current transitions, including sample 0 if present."""
    active_neg = np.asarray(waveform) < -eps
    if active_neg.size == 0:
        return 0
    return int(np.count_nonzero(active_neg & np.r_[True, ~active_neg[:-1]]))


def draw_gamma_times(freq, cv, duration_s, rng, spec: StimSpec,
                     first_pulse_sample: int = 1, max_tries: int = 2000):
    """
    Draw exactly N = round(freq * duration_s) pulse starts on the waveform grid.

    Generated trains start at `first_pulse_sample` so downstream onset detectors
    see a 0 -> negative transition, enforce the full pulse-width minimum IPI,
    and keep every pulse fully inside the stimulation window.
    """
    if freq <= 0:
        return np.empty(0, float)

    expected_n = int(round(freq * duration_s))
    if expected_n <= 0:
        return np.empty(0, float)

    first_idx = int(first_pulse_sample)
    first_time = first_idx / float(spec.fs)
    last_start_time = (spec.stim_samples - spec.pulse_samples) / float(spec.fs)
    if first_idx < 0 or first_time > last_start_time:
        raise ValueError("First pulse sample is outside the valid stimulation window.")

    if expected_n == 1:
        times = np.array([first_time], dtype=float)
        validate_pulse_start_indices(pulse_start_indices_from_times(times, spec), expected_n, spec)
        return times

    n_ipis = expected_n - 1
    min_ipi_s = spec.min_ipi_s
    available_span = last_start_time - first_time
    min_required_span = n_ipis * min_ipi_s
    if min_required_span > available_span + 1e-12:
        raise ValueError(
            f"Cannot fit {expected_n} pulses at {freq} Hz with {min_ipi_s:.6g}s minimum IPI."
        )

    target_mean_ipi = 1.0 / freq
    k = 1.0 / (cv**2) if cv > 0 else 1e12
    theta = target_mean_ipi / k

    for _ in range(max_tries):
        ipis = rng.gamma(shape=k, scale=theta, size=n_ipis)
        ipis = np.maximum(ipis, min_ipi_s)

        total_span = float(ipis.sum())
        if total_span > available_span:
            excess = ipis - min_ipi_s
            excess_sum = float(excess.sum())
            if excess_sum <= 0:
                continue
            scale = (available_span - min_required_span) / excess_sum
            if scale < 0:
                continue
            ipis = min_ipi_s + excess * scale

        times = first_time + np.r_[0.0, np.cumsum(ipis)]
        idx = pulse_start_indices_from_times(times, spec)
        try:
            validate_pulse_start_indices(idx, expected_n, spec)
        except ValueError:
            continue
        return times

    raise RuntimeError(
        f"Failed to generate valid grid-aligned pulse train for cv={cv}, freq={freq} Hz "
        f"after {max_tries} tries."
    )


def regular_train_times(freq: float | int, duration_s: float, spec: StimSpec | None = None,
                        first_pulse_sample: int = 1) -> np.ndarray:
    """
    Generate a strictly regular pulse train (for CV = 0).

    Parameters
    ----------
    freq : float or int
        Frequency of pulses in Hz.
    duration_s : float
        Stimulation duration in seconds.
    spec : StimSpec, optional
        If provided, align the first generated pulse to `first_pulse_sample` and
        validate exact count/spacing on the waveform grid.

    Returns
    -------
    times : np.ndarray
        Spike times (seconds) of a perfectly regular train within [0, duration_s).
    """
    if freq <= 0:
        return np.empty(0, dtype=float)

    expected_n = int(round(freq * duration_s))
    if expected_n <= 0:
        return np.empty(0, float)

    first_time = (int(first_pulse_sample) / float(spec.fs)) if spec is not None else 0.0
    period = 1.0 / freq
    times = first_time + np.arange(expected_n, dtype=float) * period

    if spec is not None:
        validate_pulse_start_indices(pulse_start_indices_from_times(times, spec), expected_n, spec)
        return times

    return times[times < duration_s]


def generate_biphasic_waveform(spike_times_s: np.ndarray, spec: StimSpec) -> np.ndarray:
    """
    Convert spike times (seconds) into a sampled biphasic current waveform.

    Pulse template:
      - Negative phase:  neg_samples at -amp_uA (-1 uA)
      - Interphase:      interphase_samples at 0
      - Positive phase:  pos_samples at +amp_uA (+1 uA)

    All pulses are placed on a 20 kHz time grid. Generated pulse trains are
    validated before this function is called so pulses fit fully inside the
    2 s stimulation window without overlap.

    Parameters
    ----------
    spike_times_s : np.ndarray
        Spike times in seconds (within the stim window).
    spec : StimSpec
        Sampling and pulse parameters.

    Returns
    -------
    y : np.ndarray
        1D waveform of length spec.total_samples with pulses during the first
        spec.stim_samples and zeros afterward (post-silence).
    """
    amp_neg = -spec.amp_uA
    amp_pos = +spec.amp_uA

    y = np.zeros(spec.total_samples, dtype=np.float32)
    pulse = np.concatenate(
        [
            np.full(spec.neg_samples, amp_neg, dtype=np.float32),
            np.zeros(spec.interphase_samples, dtype=np.float32),
            np.full(spec.pos_samples, amp_pos, dtype=np.float32),
        ]
    )

    L = pulse.size
    last = spec.stim_samples
    idx = pulse_start_indices_from_times(spike_times_s, spec)

    for i0 in idx:
        i1 = i0 + L
        if i1 > last:
            raise ValueError("Pulse extends beyond stimulation window; validate times before waveform generation.")
        y[i0:i1] += pulse

    return y


def make_bundle_for_pair(cv: float, freq: float | int, spec: StimSpec, slack_base: str, seed: int
                         ) -> tuple[tuple[float, float], Dict[str, np.ndarray]]:
    """
    Generate the full set of 100 waveforms for a single (cv, freq) pair.

    For each pair:
      - rep001_slack      : Slack waveform for CV>0; compatibility label for canonical CV0
      - rep002_...rep100_ : 99 additional generated waveforms
          * CV == 0.0 -> regular pulse train
          * CV > 0.0  -> gamma-based IPI train

    Returns
    -------
    key : (float, float)
        (cv, freq) pair used as a key in the global results dict.
    waveforms : dict[str, np.ndarray]
        Mapping from label -> waveform array (float32, length spec.total_samples).
    """
    waveforms: Dict[str, np.ndarray] = {}

    # ---------- 1) Establish rep001 ----------
    if cv == 0.0:
        # CV0 is deterministic. Use one exact-periodic representation for all
        # 100 repetitions; the rep001_slack name is retained only so existing
        # joins and plotting conventions do not need a schema migration.
        canonical_times = regular_train_times(
            freq, spec.stim_dur_s, spec=spec, first_pulse_sample=1
        )
        canonical_cv0 = generate_biphasic_waveform(canonical_times, spec)
        waveforms["rep001_slack"] = canonical_cv0.copy()
    else:
        slack_file = slack_path_format(cv, freq, slack_base)
        if os.path.exists(slack_file):
            try:
                waveforms["rep001_slack"] = load_slack_waveform_1uA(slack_file, spec)
            except Exception as e:
                # If Slack load fails, fall back to silence but warn
                print(f"[WARN] Failed Slack load for cv={cv}, f={freq}: {e}")
                waveforms["rep001_slack"] = np.zeros(spec.total_samples, dtype=np.float32)
        else:
            # If the Slack file is missing, warn and use silence
            print(f"[WARN] Slack file missing: {slack_file}")
            waveforms["rep001_slack"] = np.zeros(spec.total_samples, dtype=np.float32)

    # ---------- 2) Generate 99 additional patterns ----------
    rng = np.random.default_rng(seed)

    for i in range(2, 101):
        if cv == 0.0:
            # CV=0 -> reuse the already rendered canonical train
            wf = canonical_cv0.copy()
            tag = "regular"
        else:
            # CV>0 -> gamma-based spike times
            times = draw_gamma_times(freq, cv, spec.stim_dur_s, rng, spec, first_pulse_sample=1)
            wf = generate_biphasic_waveform(times, spec)
            tag = "gamma"

        expected_n = int(round(float(freq) * spec.stim_dur_s))
        realized_n = count_negative_phase_onsets(wf)
        if realized_n != expected_n:
            raise RuntimeError(
                f"Generated {tag} waveform {i} for cv={cv}, freq={freq} has "
                f"{realized_n} detected pulses; expected {expected_n}."
            )

        # Label: rep002_regular / rep002_gamma / ... rep100_...
        label = f"rep{i:03d}_{tag}"
        waveforms[label] = wf

    # Encourage GC in worker processes (if using parallel mode)
    gc.collect()

    return (cv, float(freq)), waveforms

def save_pair(out_dir: str, key: tuple[float, float], waveforms: Dict[str, np.ndarray],
    spec: StimSpec) -> Dict[str, str]:
    """
    Save the waveforms and metadata for a single (cv, freq) pair.

    Files created:
      - cv{cv:.1f}_f{freq}_waveforms.npz
      - cv{cv:.1f}_f{freq}_meta.json

    Returns
    -------
    paths : dict
        Dictionary containing paths to the npz and meta files.
    """
    cv, freq = key

    # Ensure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    # Base name for this (cv, freq) pair
    base = f"cv{cv:.1f}_f{int(freq)}"

    # 1) Save waveforms as compressed NPZ
    npz_path = os.path.join(out_dir, base + "_waveforms.npz")
    np.savez_compressed(npz_path, **waveforms) # type: ignore

    # 2) Save metadata as JSON
    meta = {
        "cv": float(cv),
        "freq_hz": float(freq),
        "fs": spec.fs,
        "stim_dur_s": spec.stim_dur_s,
        "post_silence_s": spec.post_silence_s,
        "amp_uA": spec.amp_uA,
        "neg_samples": spec.neg_samples,
        "interphase_samples": spec.interphase_samples,
        "pos_samples": spec.pos_samples,
        "pulse_samples": spec.pulse_samples,
        "min_ipi_s": spec.min_ipi_s,
        "generated_first_pulse_sample": 1,
        "generated_exact_pulse_count": True,
        "cv0_waveform_policy": "canonical_exact_periodic_all_repetitions" if cv == 0.0 else "not_applicable",
        "n_waveforms": len(waveforms),
        "npz_path": f"data/time-patterns/population/{base}_waveforms.npz",
    }

    meta_path = os.path.join(out_dir, base + "_meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return {"npz": f"data/time-patterns/population/{base}_waveforms.npz",
            "meta": f"data/time-patterns/population/{base}_meta.json"}

def generate_all_time_patterns(cv_vals, freq_vals, pw, stim_duration, slack_base, out_dir,
    parallel: bool = True, base_seed: int = 123) -> dict[str, str]:
    """
    Run the full generation pipeline over all (cv, freq) combinations.

    Parameters
    ----------
    cv_vals : iterable of float
        List of CV values to process.
    freq_vals : iterable of float/int
        List of frequencies (Hz) to process.
    pw : float
        Pulse width in seconds.
    stim_duration : float
        Stimulation duration in seconds (e.g., 2.0).
    slack_base : str
        Directory path where Slack `.txt` files live.
    out_dir : str
        Directory path where outputs will be stored.
    parallel : bool
        If True, use ThreadPoolExecutor across CPU cores; else run serially.
    base_seed : int
        Base seed used to derive per-pair seeds for reproducibility.

    Returns
    -------
    info : dict
        Contains:
          - "manifest_csv": path to the global manifest CSV
          - "out_dir":      output directory used
    """
    # Stimulation spec shared by all (cv, freq) pairs
    spec = StimSpec(
        fs=20000.0,                 # 20 kHz sampling
        stim_dur_s=float(stim_duration),
        post_silence_s=1.0,          # 1 s silence after 2 s stim
        amp_uA=1.0,                  # ±1 µA
        neg_samples=int(round(20000*(450e-6-5e-5)/2)), # 4 samples at -1 µA
        interphase_samples=1,        # 1 sample at 0
        pos_samples=int(round(20000*(450e-6-5e-5)/2)), # 4 samples at +1 µA
    )

    # Build full list of (cv, freq) pairs
    pairs = [(float(cv), float(freq)) for cv in cv_vals for freq in freq_vals]

    # Assign a unique seed to each pair, derived from base_seed
    seeds = np.random.default_rng(base_seed).integers(
        0, 2**32 - 1, size=len(pairs), dtype=np.uint64
    )

    # This dict will hold all generated waveforms keyed by (cv, freq)
    results: dict[tuple[float, float], dict[str, np.ndarray]] = {}

    if parallel:
        # ---------------- Parallel path (THREADS) ----------------
        # ThreadPoolExecutor works reliably inside notebooks & on Windows.
        with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
            # Submit each (cv, freq) job to the thread pool
            futures = {
                ex.submit(
                    make_bundle_for_pair,
                    cv,
                    freq,
                    spec,
                    slack_base,
                    int(seed),
                ): (cv, freq)
                for (cv, freq), seed in zip(pairs, seeds)
            }

            # Collect results as they finish
            for fut in as_completed(futures):
                key, bundle = fut.result()
                results[key] = bundle
    else:
        # ---------------- Serial path (debug-friendly) ----------------
        for (cv, freq), seed in zip(pairs, seeds):
            key, bundle = make_bundle_for_pair(cv, freq, spec, slack_base, int(seed))
            results[key] = bundle

    # Make sure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    # Build a manifest of all (cv, freq) outputs
    manifest_rows = []
    for key, bundle in sorted(results.items()):
        # Save this pair's waveforms & metadata
        paths = save_pair(out_dir, key, bundle, spec)
        cv, freq = key
        manifest_rows.append(
            {
                "cv": cv,
                "freq": freq,
                "npz": paths.get("npz", ""),
                "meta": paths.get("meta", ""),
            }
        )

    # Write global manifest CSV
    manifest_csv = os.path.join(out_dir, "manifest.csv")
    with open(manifest_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["cv", "freq", "npz", "meta"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    return {"manifest_csv": manifest_csv, "out_dir": out_dir}

In [ ]:
# ---------------- Generate All Time Patterns ----------------

info = generate_all_time_patterns(
    cv_vals=cv_vals,
    freq_vals=freq_vals,
    pw=pw,
    stim_duration=stim_duration,
    slack_base=slack_base,
    out_dir=out_dir,
    parallel=parallel,
    base_seed=base_seed,
)

print("Done.")
print("Manifest CSV:", info["manifest_csv"])
print("Output directory:", info["out_dir"])

In [ ]:
# ---------------- Visualize Results ----------------

# ---- Choose which (cv, freq) pair to inspect ----
cv_test   = 0.5    # choose a CV > 0 to see gamma-generated patterns
freq_test = 50    # Hz

# Build base filename pattern used in save_pair()
base_name = f"cv{cv_test:.1f}_f{int(freq_test)}"

npz_path  = os.path.join(out_dir, base_name + "_waveforms.npz")
meta_path = os.path.join(out_dir, base_name + "_meta.json")

print("Loading:")
print("  NPZ :", npz_path)
print("  META:", meta_path)

# ---- Load metadata to get sampling info ----
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

fs           = meta["fs"]           # sampling frequency (Hz), e.g., 20_000
stim_dur_s   = meta["stim_dur_s"]   # stimulation duration, e.g., 2.0
stim_samples = int(round(fs * stim_dur_s))

# ---- Load all waveforms for this (cv, freq) ----
data = np.load(npz_path)

# Historical rep001 label (canonical exact-periodic rather than Slack when CV=0)
slack = data["rep001_slack"]

# Identify gamma-generated keys (or regular if cv=0.0)
gamma_like_keys = [k for k in data.files if "gamma" in k]
if not gamma_like_keys:
    # Fallback for CV=0: just use the "regular" generated waveforms
    gamma_like_keys = [k for k in data.files if "regular" in k]

# Sort keys so the selection is deterministic, then pick up to 3
gamma_like_keys = sorted(gamma_like_keys)[:3]
gamma_waveforms = [data[k] for k in gamma_like_keys]

print(f"Found {len(gamma_like_keys)} generated patterns to plot:")
for k in gamma_like_keys:
    print("  ", k)

# ---- Build time axis for plotting ----
total_samples = slack.size             # should be 3 s * fs
t = np.arange(total_samples) / fs      # time vector for full waveform (stim + silence)

# Time axis for the stimulation window only (first 2 s)
t_stim = t[:stim_samples]

# ---- Create plots ----
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 1) Full 3-second waveform (Slack + generated)
axes[0].plot(t, slack, label="Slack (rep001)", linewidth=1.0)

for wf in gamma_waveforms:
    axes[0].plot(t, wf, alpha=0.3)

axes[0].set_title(f"Full waveform (3 s) — cv={cv_test}, freq={freq_test} Hz")
axes[0].set_ylabel("Current (µA)")
axes[0].set_xlim(0, t[-1])
axes[0].grid(True)

# 2) Zoomed-in view (first 0.2 s) to see pulse structure
zoom_end = 0.2  # seconds
zoom_mask = t <= zoom_end

axes[1].plot(t[zoom_mask], slack[zoom_mask], label="Slack (rep001)", linewidth=1.0)

for wf in gamma_waveforms:
    axes[1].plot(t[zoom_mask], wf[zoom_mask], alpha=0.5)

axes[1].set_title("Zoomed waveform (0–0.2 s)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Current (µA)")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()